# YaSpeech: автопротоколирование деловых встреч на Yandex Cloud
### SpeechKit (ASR) + YandexGPT + Object Storage

Кукбук показывает, как собрать сервис **«запись встречи → готовый протокол»** на трёх сервисах Yandex Cloud.


# 1. Введение

## Назначение

В этом кукбуке демонстрируется, как собрать сервис **«запись встречи → готовый протокол»** с помощью:
- **SpeechKit STT v3** — асинхронное распознавание речи с разметкой по каналам
- **YandexGPT** — диаризация по составу команды, коррекция ошибок ASR, генерация протокола
- **Object Storage** — промежуточное хранение аудио (обязательно: SpeechKit async читает файл только по ссылке)

## Описание задачи

**Бизнес-задача**: команды, которые фиксируют решения и задачи по итогам планёрок, тратят время на ручное протоколирование. Этот кукбук демонстрирует:

- Распознавание речи из аудиозаписи встречи
- Коррекцию типичных ошибок ASR (разорванные слова, аббревиатуры, произнесённые словами числа)
- Генерацию структурированного протокола: решения, задачи с ответственными, открытые вопросы

**Основные сервисы Yandex Cloud:**
- SpeechKit STT v3
- AI Studio с YandexGPT
- Object Storage

***

## Ожидаемый результат

Готовый пайплайн «аудио → протокол» — рабочий каркас, от которого можно оттолкнуться и построить решение под свою задачу.


# 2. Архитектура решения

## Компоненты системы

```
Аудио (.wav/.ogg/.mp3)
      │
      ▼
Object Storage (SpeechKit читает аудио только по ссылке)
      │
      ▼
SpeechKit STT v3 — асинхронное распознавание
      │
      ▼
Один вызов YandexGPT:
  диаризация по составу команды + коррекция ASR + протокол
      │
      ▼
Структурированный протокол (JSON)
```

| Компонент | Назначение | Роли и права |
|---|---|---|
| **SpeechKit STT v3** | Распознавание речи (`recognizeFileAsync` + `getRecognition`) | `ai.speechkit-stt.user` — вызов распознавания |
| **YandexGPT** | Один вызов: разметка спикеров по составу команды, коррекция ошибок ASR, протокол | `ai.languageModels.user` — генерация текста |
| **Object Storage** | Промежуточное хранение аудио для SpeechKit | `storage.editor` — создание бакета и загрузка объектов |

## Описание ролей и их области действия

**Сервисный аккаунт** — учётная запись, от имени которой приложения и автоматизированные сервисы обращаются к ресурсам Yandex Cloud. В отличие от обычных пользовательских аккаунтов, сервисные аккаунты используются для программного доступа и не требуют браузерной аутентификации.

- `ai.speechkit-stt.user` — *область действия: каталог и вложенные ресурсы.* Позволяет отправлять запросы на распознавание речи в SpeechKit (`recognizeFileAsync`, `getRecognition`).
- `ai.languageModels.user` — *область действия: каталог и вложенные ресурсы.* Минимальная роль для работы с моделями генерации текста YandexGPT — отправка запросов на генерацию.
- `storage.editor` — *область действия: каталог, вложенные ресурсы, либо конкретный бакет.* Даёт право создавать бакет и загружать в него объекты — обе операции нужны коду кукбука (`ensure_bucket`, `upload_audio`). Более узкая роль `storage.uploader` разрешает только загрузку в уже существующий бакет, но не его создание — если бакет уже создан заранее, можно использовать её вместо `storage.editor`.

## Как назначить роль сервисному аккаунту через консоль Yandex Cloud

1. Откройте сервис Identity and Access Management (IAM) в консоли управления.
2. Выберите каталог, к которому нужно предоставить доступ.
3. Перейдите на вкладку «Права доступа» → «Настроить доступ».
4. Выберите раздел «Сервисные аккаунты», найдите нужный или создайте новый.
5. Нажмите «Добавить роль» и выберите роли из списка выше — можно назначить несколько ролей сразу.
6. Сохраните изменения.


# 3. Подготовка окружения

Понадобятся три вещи:
- **FOLDER_ID** — идентификатор каталога: https://yandex.cloud/ru/docs/resource-manager/operations/folder/get-id
- **YC_API_KEY** — ключ сервисного аккаунта с ролями `ai.speechkit-stt.user`, `ai.languageModels.user`, `storage.editor` (см. раздел 2): https://yandex.cloud/ru/docs/iam/operations/api-key/create
- **AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY** — статические ключи для Object Storage: https://yandex.cloud/ru/docs/iam/concepts/authorization/access-key


Устанавливаем нужные библиотеки.


In [ ]:
!pip install -q openai python-dotenv boto3 requests pydantic


In [ ]:
import os
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import boto3
import requests
import openai
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field, field_validator

load_dotenv(find_dotenv())

FOLDER_ID = os.getenv("FOLDER_ID", "YOUR_FOLDER_ID")
YC_API_KEY = os.getenv("YC_API_KEY", "YOUR_API_KEY")
GPT_MODEL = os.getenv("GPT_MODEL", "yandexgpt-5-lite")  # см. актуальный список моделей: https://yandex.cloud/en/docs/ai-studio/concepts/generation/models
MODEL_URI = f"gpt://{FOLDER_ID}/{GPT_MODEL}"

# Responses API — тот же OpenAI-совместимый клиент, но со своим base_url и
# поддержкой structured/tool-калбэков (см. https://aistudio.yandex.ru/docs/en/ai-studio/concepts/api.html)
gpt_client = openai.OpenAI(
    api_key=YC_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
)

S3_BUCKET = os.getenv("BUCKET_NAME", "YOUR_BUCKET_NAME")
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "YOUR_AWS_ACCESS_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "YOUR_AWS_SECRET_KEY"),
    region_name="ru-central1",
)

SPEECHKIT_RECOGNIZE_URL = "https://stt.api.cloud.yandex.net/stt/v3/recognizeFileAsync"
SPEECHKIT_GET_URL = "https://stt.api.cloud.yandex.net/stt/v3/getRecognition"
OPERATION_URL = "https://operation.api.cloud.yandex.net/operations"

print("Клиенты инициализированы:", MODEL_URI)


# 4. Загрузка аудио в Object Storage

SpeechKit async принимает аудио **только по ссылке** на Object Storage — поэтому bucket и upload обязательны, даже в учебном примере.

Поддерживаемые контейнеры: **WAV, OggOpus, MP3** (без M4A) — конвертация формата ниже сделана через `ffmpeg` автоматически. Отдельное требование именно для `speakerLabeling`: SpeechKit **разметку дикторов делает только для моно** (1 канал) — на стерео-файле весь запрос упадёт с ошибкой `Speaker labeling is only for mono-channel audio files`, поэтому ячейка ниже всегда сводит звук в 16 кГц / 1 канал, даже если исходник уже был WAV.


In [ ]:
import subprocess

def ensure_mono_wav(local_path: str) -> str:
    """SpeechKit со speakerLabeling принимает только моно (см. markdown выше) —
    сводим любой входной WAV/MP3/OggOpus в WAV 16кГц/1 канал через ffmpeg,
    даже если формат контейнера уже поддерживается как есть."""
    out_path = str(Path(local_path).with_suffix(".mono.wav"))
    cmd = ["ffmpeg", "-y", "-i", local_path, "-ar", "16000", "-ac", "1", out_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg не смог свести {local_path} в моно: {result.stderr[-1000:]}")
    print(f"Сведено в моно: {out_path}")
    return out_path


def ensure_bucket(bucket_name: str) -> None:
    existing = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
    if bucket_name not in existing:
        s3.create_bucket(Bucket=bucket_name)
    print(f"Бакет готов: {bucket_name}")


def upload_audio(local_path: str, key: str) -> str:
    s3.upload_file(local_path, S3_BUCKET, key)
    uri = f"https://storage.yandexcloud.net/{S3_BUCKET}/{key}"
    print(f"Загружено: {uri}")
    return uri


ensure_bucket(S3_BUCKET)


# 5. SpeechKit STT v3: асинхронное распознавание

`recognizeFileAsync` возвращает `id` операции сразу, распознавание идёт в фоне — опрашиваем `operation.api` до `done: true`, затем забираем результат через `getRecognition` (NDJSON, по одному объекту на строку).

С включённым `speakerLabeling` для монозаписи каждое значение `channelTag` в результате соответствует одному диктору (до двух дикторов на моно). На практике акустическая разметка не всегда чистая: если оба собеседника говорят в один микрофон, границы реплик могут получиться шумными. В этом кукбуке отдаём как исходную разметку, так и весь текст в LLM — модель может уточнить границы реплик и присвоить реальные имена по составу команды.

Одна минута аудио распознаётся примерно за 10 секунд (см. [пошаговую инструкцию](https://aistudio.yandex.ru/docs/ru/speechkit/stt/api/transcribation-api-v3)).


In [ ]:
def start_recognition(audio_uri: str, language: str = "ru-RU") -> str:
    headers = {"Authorization": f"Api-Key {YC_API_KEY}", "Content-Type": "application/json"}
    body = {
        "uri": audio_uri,
        "recognitionModel": {
            "model": "general",
            "audioFormat": {"containerAudio": {"containerAudioType": "WAV"}},
            "textNormalization": {
                "textNormalization": "TEXT_NORMALIZATION_ENABLED",
                "profanityFilter": False,
                "literatureText": True,
            },
            "languageRestriction": {"restrictionType": "WHITELIST", "languageCode": [language]},
        },
        "speakerLabeling": {"speakerLabeling": "SPEAKER_LABELING_ENABLED"},
    }
    resp = requests.post(SPEECHKIT_RECOGNIZE_URL, headers=headers, json=body, timeout=30)
    resp.raise_for_status()
    operation_id = resp.json()["id"]
    print(f"Операция запущена: {operation_id}")
    return operation_id


def wait_operation(operation_id: str, poll_interval_s: int = 5, max_wait_s: int = 900) -> None:
    headers = {"Authorization": f"Api-Key {YC_API_KEY}"}
    waited = 0
    while waited < max_wait_s:
        resp = requests.get(f"{OPERATION_URL}/{operation_id}", headers=headers, timeout=15)
        resp.raise_for_status()
        op = resp.json()
        if op.get("done"):
            if op.get("error"):
                raise RuntimeError(f"SpeechKit error: {op['error']}")
            return
        time.sleep(poll_interval_s)
        waited += poll_interval_s
    raise TimeoutError("SpeechKit: не завершилось за отведённое время")


def fetch_recognition(operation_id: str) -> List[Dict[str, Any]]:
    """Каждый чанк: {channelTag, text} — channelTag соответствует диктору
    при включённом speakerLabeling (моно, до 2 дикторов)."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}"}
    resp = requests.get(SPEECHKIT_GET_URL, headers=headers,
                        params={"operationId": operation_id}, timeout=60)
    resp.raise_for_status()

    chunks = []
    for line in resp.text.splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        final = obj.get("result", {}).get("final")
        if not final:
            continue
        alternatives = final.get("alternatives", [])
        if not alternatives or not alternatives[0].get("text", "").strip():
            continue
        chunks.append({"channelTag": final.get("channelTag", "0"), "text": alternatives[0]["text"].strip()})
    return chunks


def recognize_meeting_audio(audio_uri: str) -> List[Dict[str, Any]]:
    operation_id = start_recognition(audio_uri)
    wait_operation(operation_id)
    return fetch_recognition(operation_id)


## 5.1 Постобработка транскрипта

Склеиваем подряд идущие реплики одного канала и фильтруем короткие слова-паразиты.


In [ ]:
NOISE_MARKERS = {"угу", "ага", "эм", "эээ", "ну", "вот", "это", "так"}


def is_noise_segment(text: str) -> bool:
    words = text.lower().split()
    return bool(words) and len(words) <= 3 and all(w in NOISE_MARKERS for w in words)


def postprocess_transcript(chunks: List[Dict[str, Any]]) -> str:
    """Склеивает реплики одного канала подряд, фильтрует мусор, отдаёт
    один текст с метками [Канал N] — окончательную разметку спикеров
    (диаризацию по именам) делает LLM-проход ниже."""
    filtered = [c for c in chunks if c["text"] and not is_noise_segment(c["text"])]

    merged: List[Dict[str, str]] = []
    for c in filtered:
        if merged and merged[-1]["channel"] == c["channelTag"]:
            merged[-1]["text"] += " " + c["text"]
        else:
            merged.append({"channel": c["channelTag"], "text": c["text"]})

    return "\n".join(f"[Канал {m['channel']}] {m['text']}" for m in merged)


# 6. Один вызов YandexGPT: спикеры + коррекция + протокол

Вместо цепочки узких проходов — один структурированный запрос: модель получает сырой транскрипт (с грубой разметкой по каналам) и состав команды проекта, а возвращает исправленный текст с реальными именами и готовый протокол.

Три инструкции, которые здесь особенно важны:
- **не выдумывай** — пустой список лучше вымышленного решения или задачи;
- **не меняй числа, даты и суммы** при исправлении ошибок распознавания;
- **разметка по каналам — подсказка, не факт** — модель может перегруппировать реплики, если разметка SpeechKit похожа на шум.


In [ ]:
class ActionItem(BaseModel):
    owner: str = Field(description="Ответственный, или 'Команда' если не назван")
    task: str
    deadline: Optional[str] = None


class ProtocolOutput(BaseModel):
    """Pydantic-модель не диктует формат ответа модели (Responses API это
    делает текстовым промптом, см. build_protocol_prompt), а валидирует то,
    что пришло — сразу видно, если модель что-то упустила или исказила тип."""
    corrected_transcript: str = Field(description="Транскрипт с исправленными ошибками ASR и реальными именами спикеров вместо 'Канал N'")
    meeting_title: str
    domain: str = Field(description="Предметная сфера встречи, например: строительство, IT, продажи")
    summary: str = Field(description="4-6 предложений деловой прозы")
    decisions: List[str] = Field(default_factory=list, description="Только реально принятые решения, дословно из транскрипта")
    action_items: List[ActionItem] = Field(default_factory=list)
    open_questions: List[str] = Field(default_factory=list, description="Вопросы, которые обсуждали, но не решили")

    @field_validator("meeting_title", mode="before")
    @classmethod
    def _fallback_title(cls, v):
        """Промпт просит модель никогда не оставлять meeting_title пустым (см.
        build_protocol_prompt, п.4), но на коротких/малоинформативных встречах
        модель иногда всё же возвращает null, применяя к заголовку правило
        "не выдумывай" — программная страховка на этот случай, а не только
        промпт-инструкция."""
        return v if v else "Встреча без определённой темы"


In [ ]:
def build_protocol_prompt(project_team: List[Dict[str, str]]) -> str:
    """В промпт кладём готовый ПРИМЕР ответа с реальными значениями, а не дамп
    JSON Schema (properties/type/description) — модель на практике путает такую
    схему с данными и подставляет метаданные полей вместо самих значений."""
    team_block = "\n".join(f"- {m['name']} ({m.get('role', 'без роли')})" for m in project_team) \
        or "Состав участников неизвестен — используй 'Спикер 1', 'Спикер 2' и т.д."

    example = {
        "corrected_transcript": "Иванов Алексей: Коллеги, начнём с фундамента...",
        "meeting_title": "Планёрка по объекту на Садовой",
        "domain": "строительство",
        "summary": "4-6 предложений деловой прозы.",
        "decisions": ["решение дословно из транскрипта"],
        "action_items": [{"owner": "Иванов Алексей", "task": "конкретная задача", "deadline": "2026-08-15 или null"}],
        "open_questions": ["вопрос, который обсуждали, но не решили"],
    }
    example_json = json.dumps(example, ensure_ascii=False, indent=2)

    return f"""Ты — секретарь деловых встреч. Тебе дан транскрипт, полученный автоматическим
распознаванием речи (ASR), с грубой разметкой по каналам записи ([Канал 0], [Канал 1], ...).

УЧАСТНИКИ ВСТРЕЧИ (известный состав):
{team_block}

ЗАДАЧИ:
1. Разметь реплики по реальным участникам. Разметка по каналам — подсказка о числе
   говорящих, а не факт: если она выглядит нарезанной через каждые несколько слов,
   перегруппируй реплики по смыслу (вопрос → ответ, обращение по имени, смена темы).
   Если имя не удаётся определить уверенно — оставь метку "Спикер N".
2. Исправь искажения ASR: разорванные слова, аббревиатуры и марки, произнесённые
   словами ("м триста пятьдесят" → "М350"). Пунктуация и заглавные буквы — по смыслу.
   НИКОГДА не меняй сами числа, даты и суммы — только форму записи.
3. Составь протокол: реальные решения ("решили", "утвердили"), задачи с ответственным
   и дедлайном (если не назван — "Команда" и null), открытые вопросы, которые
   обсуждали, но не решили.
4. "meeting_title" — ВСЕГДА строка, никогда null: если из транскрипта не ясна
   конкретная тема встречи, используй общее описание по домену и участникам
   (например "Планёрка по текущим задачам"), а не оставляй поле пустым.

НЕ ВЫДУМЫВАЙ факты. Если решений или задач нет в тексте — верни пустой список,
а не пример "для порядка". Это правило не относится к meeting_title (см. п.4) —
заголовок обязателен как ярлык, даже когда тема расплывчата.

Верни ТОЛЬКО корректный JSON без markdown-обёртки, СТРОГО в этом формате
(ниже — форма ответа с примерами значений, не переписывай примеры дословно):
{example_json}"""


In [ ]:
def parse_protocol_json(raw_text: str) -> ProtocolOutput:
    """Модель иногда оборачивает JSON в ```json ... ``` — снимаем обёртку перед парсингом."""
    candidate = raw_text.strip()
    if candidate.startswith("```"):
        candidate = candidate.split("```")[1]
        candidate = candidate[4:].strip() if candidate.lower().startswith("json") else candidate.strip()
    return ProtocolOutput.model_validate_json(candidate)


def extract_meeting_protocol(raw_transcript: str, project_team: List[Dict[str, str]]) -> ProtocolOutput:
    system_prompt = build_protocol_prompt(project_team)

    response = gpt_client.responses.create(
        model=MODEL_URI,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Транскрипт:\n{raw_transcript[:20000]}"},
        ],
        temperature=0.2,
        max_output_tokens=8000,  # ответ включает исправленный транскрипт целиком — на длинных встречах может не хватить и этого, см. раздел 9
    )
    return parse_protocol_json(response.output_text)


# 7. Сборка пайплайна


In [ ]:
def run_meeting_pipeline(audio_local_path: str, project_team: List[Dict[str, str]]) -> Tuple[ProtocolOutput, str]:
    """Возвращает (протокол, meeting_id) — id нужен, чтобы сохранить результат
    прогона в файл с тем же именем, что и аудио в Object Storage."""
    meeting_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    mono_path = ensure_mono_wav(audio_local_path)
    audio_uri = upload_audio(mono_path, f"audio/{meeting_id}.wav")

    raw_chunks = recognize_meeting_audio(audio_uri)
    raw_text = postprocess_transcript(raw_chunks)
    print(f"Транскрипт: {len(raw_text)} символов")

    protocol = extract_meeting_protocol(raw_text, project_team)
    return protocol, meeting_id


def save_transcript_to_txt(protocol: ProtocolOutput, meeting_id: str) -> str:
    """Сохраняет только исправленный транскрипт — отдельным файлом от протокола,
    рядом с ноутбуком (текущая рабочая директория)."""
    out_path = f"transcript_{meeting_id}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(protocol.corrected_transcript + "\n")
    print(f"Сохранено: {out_path}")
    return out_path


def save_protocol_to_txt(protocol: ProtocolOutput, meeting_id: str) -> str:
    """Сохраняет только протокол (JSON) — отдельным файлом от транскрипта."""
    out_path = f"protocol_{meeting_id}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(protocol.model_dump(), ensure_ascii=False, indent=2) + "\n")
    print(f"Сохранено: {out_path}")
    return out_path


# 8. Тестирование на реальном аудиофайле

Замените `AUDIO_LOCAL_PATH` на свой WAV-файл и впишите реальных участников встречи, чтобы увидеть диаризацию по именам.


In [ ]:
AUDIO_LOCAL_PATH = "your_meeting_recording.wav"  # <-- впишите путь к своему аудиофайлу (WAV/OggOpus/MP3)

project_team = [
    {"name": "Иванов Алексей", "role": "прораб"},
    {"name": "Петрова Мария", "role": "заказчик"},
    {"name": "Сидоров Игорь", "role": "инженер ПТО"},
]


In [ ]:
protocol, meeting_id = run_meeting_pipeline(AUDIO_LOCAL_PATH, project_team)


In [ ]:
print("=" * 80)
print("ИСПРАВЛЕННЫЙ ТРАНСКРИПТ")
print("=" * 80)
print(protocol.corrected_transcript)

save_transcript_to_txt(protocol, meeting_id)


In [ ]:
print("=" * 80)
print("ПРОТОКОЛ")
print("=" * 80)
print(json.dumps(protocol.model_dump(), ensure_ascii=False, indent=2))

save_protocol_to_txt(protocol, meeting_id)


# 9. Результаты и анализ

## Что мы достигли

Полный цикл «аудио → протокол» на трёх сервисах Yandex Cloud за один структурированный вызов LLM:
- аудио сведено в моно и загружено в Object Storage;
- SpeechKit вернул транскрипт с разметкой по каналам;
- YandexGPT одним вызовом расставил реальные имена участников, исправил ошибки ASR и собрал протокол — решения, задачи с ответственными, открытые вопросы;
- транскрипт и протокол сохранены отдельными файлами (`transcript_<id>.txt`, `protocol_<id>.txt`).

## Как это работает

Диаризация здесь не полагается на акустическую разметку SpeechKit как на последнюю инстанцию: модель получает и разметку по каналам, и состав команды проекта, и сама решает, где реально сменился говорящий — по прямым обращениям, характерной лексике роли, структуре диалога (вопрос → ответ). Программная страховка (`field_validator` в `ProtocolOutput`) подхватывает случаи, когда модель всё же нарушает инструкцию промпта — например, возвращает `null` вместо обязательного заголовка встречи.

## Ключевые параметры

- `temperature=0.2` — консервативная настройка для задачи, где важна точность (коррекция чисел, дат), а не творческое разнообразие.
- `max_output_tokens=8000` — ответ включает исправленный транскрипт целиком, поэтому лимит выше типового; на длинных встречах может не хватить и этого (см. ограничения ниже).
- `GPT_MODEL=yandexgpt-5-lite` — выбрана по цене/скорости для учебного примера; для более сложных доменов сравните с `yandexgpt-5-pro` на своих записях.

## Ограничения учебной версии

- Один вызов на всю встречу упирается в контекст модели на длинных записях (десятки тысяч символов). Для часовых и более длинных встреч стоит разбить на этапы — например, диаризацию и коррекцию отдельно от генерации протокола, с чанкингом транскрипта по репликам.
- Модель может пропустить реплику при большом объёме входного текста — для прод-сценария стоит добавить проверку целостности (построчную нумерацию реплик и сверку, что каждая вернулась) и программный валидатор чисел/дат вместо расчёта только на промпт-инструкцию.
- Диаризация иногда дробит одну связную мысль говорящего на несколько коротких реплик разных «спикеров» — особенно на репликах-подтверждениях («да», «понял», «хорошо»). Если это критично для вашего сценария, стоит усилить инструкцию промпта или добавить пост-обработку, которая склеивает подряд идущие короткие реплики.
- Оценивайте качество коррекции на своих записях: возьмите 10-20 реальных фрагментов, аннотируйте эталонный текст руками и сравните с результатом модели, прежде чем фиксировать выбор модели в проде.

## Куда двигаться дальше

- Разнести старт распознавания и опрос операции по отдельным вызовам (Cloud Functions + очередь), если сервис асинхронный и без общего процесса ожидания.
- Добавить QA-проход, который проверяет пункты протокола на соответствие транскрипту.


# 10. Полезные ссылки

- [SpeechKit STT v3, асинхронное распознавание](https://aistudio.yandex.ru/docs/ru/speechkit/stt/api/transcribation-api-v3)
- [Поддерживаемые форматы аудио](https://aistudio.yandex.ru/docs/ru/speechkit/formats)
- [Доступные генеративные модели](https://aistudio.yandex.ru/docs/ru/ai-studio/concepts/generation/models)
- [Object Storage (S3-совместимое API)](https://yandex.cloud/ru/docs/storage/s3/)
